# 18 — Embedding design amplifies the effect

In `04/17` the equal-weight qutrit read the second harmonic, and needed longer records than the second-harmonic PLV to do it. This notebook turns a dial the capstone left at its default, the **weights** of the harmonics. You will first find on paper where each harmonic sits in the joint density and how a weight moves it. Then you will measure what it buys, and see why the gain stops where the matching classical statistic already stands. A lever of another kind, the amplitude embedding, is the subject of `04/19`.

**Prerequisites:** `16_discriminating_experiment` (the matched-PLV ensembles, the area under the curve and the detection rate), `17_capstone_synthesis` (the embedding-as-dial idea), `15_richer_embedding_applied` (the multi-frequency qutrit).

**What you'll be able to do**
- Write by hand which entries of $\rho_{AB}$ carry each circular moment.
- Score a weighting with the protocol of `04/16` beside the classical statistic of the same moment, and read a near-tie.
- Explain from the long-run response of the measure why turning the first harmonic down helps.
- Derive the closed form at $w_1 = 0$, and show that a larger gap is not a better detector.

In [ ]:
import csv
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS, CMAP_COST
from qot_course.quantum_ot.bench_protocol import (
    auc,
    detection_rate,
    harmonic_plv,
    long_run_statistics,
)
from qot_course.quantum_ot.capstone import plv, coupling_qmi
from qot_course.quantum_ot.embeddings import weighted_multifreq_state, joint_density_from_states
from qot_course.quantum_ot.discrimination import matched_plv_ensembles

np.random.seed(42)
viz.use_course_style()

# Sample count per channel: large enough that the circular moments are tight Monte-Carlo
# estimates (error ~ 1/sqrt(N)), matching the discriminating experiment of 04/16.
N = 150_000

## 1. The dial we left at its default, and where each moment sits

The multi-frequency qutrit of `04/15` is an *equal-weight* superposition, $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$. It gives the first circular moment of the phase difference (the one PLV reads) and the second (the one that differs between our two ensembles) comparable room. In the matched-PLV experiment the first moment carries no difference between the ensembles, yet it wobbles from record to record. What if we turned it down? `weighted_multifreq_state` sets the weights $(w_0, w_1, w_2)$ and rescales them to unit length, so that $p_k = w_k^2$ sums to one.

**Where each moment sits, by hand.** The entry of $\rho_{AB}$ between $|i, j\rangle$ and $|k, l\rangle$ is
$$
\langle i, j|\rho_{AB}|k, l\rangle = w_i w_j w_k w_l\,\bigl\langle e^{i[(i-k)\theta_A + (j-l)\theta_B]}\bigr\rangle .
$$
Write the exponent as $(i - k + j - l)\,\theta_A - (j - l)\,\delta$, with $\delta = \theta_A - \theta_B$ the phase difference, as in `04/11`. Since $\theta_A$ is uniform and independent of $\delta$, the average removes every entry unless $i - k = l - j$, and what survives is $w_iw_jw_kw_l\,m_{i-k}$, where $m_n = \langle e^{in\delta}\rangle$.

Now count. Four entries carry $m_1$, with the products $w_0^2w_1^2$, $w_0w_1^2w_2$ (twice) and $w_1^2w_2^2$: each has a factor $w_1^2$. One entry carries $m_2$, with $(i, k) = (2, 0)$ and $(j, l) = (0, 2)$, and its product $w_0^2w_2^2$ has no $w_1$ at all. Lowering $w_1$ quietens the first moment and leaves the room to the second.

**In numbers.** In the high ensemble of `04/16`, $m_1 = 0.4$ and $m_2 = 0.3$. Equal weights give every product $1/9$, so $\langle 1,0|\rho_{AB}|0,1\rangle = 0.4/9 = 0.044$ and $\langle 2,0|\rho_{AB}|0,2\rangle = 0.3/9 = 0.033$. The weights $(1, 0.2, 2)$ give $p = (0.198, 0.0079, 0.794)$, so the first entry falls to $0.00063$ and the second rises to $0.047$. The ratio $w_2^2/w_1^2 = 100$ is how much this weighting favours the second moment over the first.

In [ ]:
# The two ensembles of 04/16: identical first moment a1 (-> identical PLV), different
# second moment a2 (0.0 against 0.3). Section 1 reads the high one; section 2 uses both.
(ta_lo, tb_lo), (ta_hi, tb_hi) = matched_plv_ensembles(
    N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=0
)
M1, M2 = 0.4, 0.3  # target moments of the high ensemble

# |i>_A |j>_B is row 3 i + j of the 9 x 9 matrix.
M1_ENTRIES = [(3, 1), (4, 2), (6, 4), (7, 5)]  # the four entries with i - k = l - j = 1
M2_ENTRY = (6, 2)                               # <2,0| rho |0,2>, the one entry with m2


def joint_density(ta, tb, weights):
    """Joint density (9 x 9) of the weighted qutrit embedding of two phase records."""
    return joint_density_from_states(
        weighted_multifreq_state(ta, (1, 2), weights),
        weighted_multifreq_state(tb, (1, 2), weights),
    )


def correlation_part(rho):
    """rho_AB minus the product of its two marginals: what the coupling measure responds to."""
    r = rho.reshape(3, 3, 3, 3)
    rho_a, rho_b = np.einsum("ijkj->ik", r), np.einsum("ijil->jl", r)
    return rho - np.kron(rho_a, rho_b)


panels = []
for name, w in (("equal weights (1, 1, 1)", (1.0, 1.0, 1.0)), ("weights (1, 0.2, 2)", (1.0, 0.2, 2.0))):
    p = np.array(w) ** 2 / np.sum(np.array(w) ** 2)
    rho = joint_density(ta_hi, tb_hi, w)
    print(f"{name}:  p = ({p[0]:.4f}, {p[1]:.4f}, {p[2]:.4f}),  w2^2 / w1^2 = {p[2] / p[1]:.0f}")
    print(f"   <1,0|rho|0,1>  by hand p0 p1 m1 = {p[0] * p[1] * M1:.5f}   from samples {rho[3, 1].real:.5f}")
    print(f"   <2,0|rho|0,2>  by hand p0 p2 m2 = {p[0] * p[2] * M2:.5f}   from samples {rho[M2_ENTRY].real:.5f}")
    panels.append((name, np.abs(correlation_part(rho))))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
vmax = max(panel.max() for _, panel in panels)
labels = [f"{i}{j}" for i in range(3) for j in range(3)]
for ax, (name, panel) in zip(axes, panels):
    im = ax.imshow(panel, cmap=CMAP_COST, vmin=0.0, vmax=vmax)
    for (r, c) in M1_ENTRIES + [(c, r) for r, c in M1_ENTRIES]:
        ax.text(c, r, "$m_1$", ha="center", va="center", fontsize=8, color=COLORS["text"])
    for (r, c) in (M2_ENTRY, M2_ENTRY[::-1]):
        ax.text(c, r, "$m_2$", ha="center", va="center", fontsize=8, color=COLORS["text"])
    ax.set_xticks(range(9))
    ax.set_yticks(range(9))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("column $|k, l\\rangle$")
    ax.set_ylabel("row $|i, j\\rangle$")
    ax.set_title(name, pad=10)
    ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.85, label="$|\\rho_{AB} - \\rho_A \\otimes \\rho_B|$")
fig.suptitle("Where the two moments sit in the joint density (high ensemble)", y=1.0)
plt.show()

**Read the output and the figure.** The printed entries agree with the hand computation within $0.0003$: $0.0442$ and $0.0335$ against $0.0444$ and $0.0333$ for equal weights, $0.00063$ and $0.0475$ against $0.00063$ and $0.0472$ for $(1, 0.2, 2)$.

Each panel shows how far each entry of $\rho_{AB}$ is from the product of the marginals, the part a coupling measure responds to, on one shared colour scale. With equal weights (left), the first moment lights up eight cells, the four entries you counted and their mirror images, and the second moment two. With $(1, 0.2, 2)$ (right), the first-moment cells have almost faded and the two second-moment cells are the darkest of the figure; the faint cells away from the named entries are sampling noise. The data did not change; the embedding moved the light.

## 2. A larger gap between the two ensembles

We now take the quantum mutual information (QMI) of both ensembles of `04/16` and try three weightings: the equal-weight default, then $(1, 0.5, 2)$ and $(1, 0.2, 2)$, the two carried by the bench files of `04/17`. The weight 2 on the top level is part of that choice, not a requirement; section 4 shows what it does.

In [ ]:
print(f"PLV gap (matched by construction): {abs(plv(ta_lo, tb_lo) - plv(ta_hi, tb_hi)):.4f}")


def weighted_qmi(ta, tb, weights, harmonics=(1, 2)):
    """Quantum mutual information (nats) of the weighted multi-frequency joint embedding."""
    dim = 1 + len(harmonics)
    rho = joint_density_from_states(
        weighted_multifreq_state(ta, harmonics, weights),
        weighted_multifreq_state(tb, harmonics, weights),
    )
    return coupling_qmi(rho, dims=(dim, dim))


configs = [
    ("equal weights (1, 1, 1)", (1.0, 1.0, 1.0)),
    ("first harmonic at 0.5: (1, 0.5, 2)", (1.0, 0.5, 2.0)),
    ("first harmonic at 0.2: (1, 0.2, 2)", (1.0, 0.2, 2.0)),
]
gaps = []
print(f"{'weights':<38}{'low':>10}{'high':>10}{'gap':>10}   (nats)")
for name, w in configs:
    low, high = weighted_qmi(ta_lo, tb_lo, w), weighted_qmi(ta_hi, tb_hi, w)
    gaps.append(high - low)
    print(f"{name:<38}{low:>10.5f}{high:>10.5f}{high - low:>10.5f}")

**Read the output.** The first line confirms the construction: the two ensembles have the same PLV to within $0.002$. Two things move as the first-harmonic weight goes down. The values shrink, from $0.074$ to $0.001$ nat for the low ensemble, because most of the equal-weight QMI came from the first moment, the eight cells of the left panel above. And the gap grows, from $0.0044$ to $0.0145$ nat.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
y = np.arange(len(configs))[::-1]
bar_colors = [COLORS["muted"], COLORS["quantum"], COLORS["highlight"]]
ax.barh(y, gaps, color=bar_colors, edgecolor="white", linewidth=1.2, zorder=3)
# What a real device would have to resolve. A gap is a DIFFERENCE of two QMI estimates; if the
# two are independent and equally noisy, its statistical band is sqrt(2) * sigma_stat.
# Notebook 05/06 reports two values of sigma_stat for a single estimate: 0.012 nats on the
# real processor (the line drawn here) and 0.0128 nats on the offline device model.
HARDWARE_SIGMA_STAT = 0.012   # real run reported in 05/06
MODEL_SIGMA_STAT = 0.0128     # offline device model of 05/06
hardware_band = np.sqrt(2) * HARDWARE_SIGMA_STAT
model_band = np.sqrt(2) * MODEL_SIGMA_STAT
ax.axvline(hardware_band, color=COLORS["negative"], lw=1.6, ls="--", zorder=2,
           label=f"one band of the real device ({hardware_band:.3f} nat)")
for yi, g in zip(y, gaps):
    ax.annotate(f"{g:.4f}", (g, yi), xytext=(6, 0), textcoords="offset points",
                va="center", ha="left", fontsize=10, color=COLORS["text"])
ax.set_yticks(y)
ax.set_yticklabels([name for name, _ in configs])
ax.set_xlabel("QMI gap between the two matched-PLV ensembles  (nats)")
ax.set_xlim(0, max(max(gaps) * 1.3, hardware_band * 1.1))
ax.set_title("Turning the first harmonic down enlarges the gap", pad=10)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()
for (name, _), g in zip(configs, gaps):
    print(f"{name:<38} gap / band:  real device = {g / hardware_band:.2f}   "
          f"offline model = {g / model_band:.2f}")

**Read the figure.** The grey bar, the equal-weight default of `04/16`, reaches about a quarter of the way to the dashed line.

That line is one **band** of a real device: one standard deviation, caused by the finite number of shots, of a *difference* of two mutual informations. Notebook `05/06`, later in the course, measures $0.012$ nat for a single estimate on the real processor, and a difference of two independent estimates scatters $\sqrt 2$ times more.

As the first-harmonic weight goes down (blue, then rose), the gap climbs to $0.85$ of that band, or $0.80$ bands of the offline device model of `05/06`, whose scatter is about the same. Either way the weighted gap is a little under one band: a real gain, and not yet a detection, for which the usual rule of thumb asks two or three bands. The band was measured on a two-qubit probe, while a pair of qutrits needs four qubits, so this is an order of magnitude and not a forecast. `05/06` also finds that the scatter depends on the state, from 0.0045 nat on its least correlated probe to 0.0128 on the Bell pair, and that the device shrinks a gap; the ratio can move in either direction.

## 3. Does a larger gap mean fewer samples?

A gap is a statement about very long records. What you hold in practice is one record of finite length, and `04/16` gave you two ways to say whether a statistic tells two ensembles apart at that length. The **area under the curve** (AUC) is the probability that a high ensemble scores above a low one: $0.5$ is chance, $1$ is perfect ranking. The **detection rate** is the share of high-minus-low differences that fall outside the **detection band**, the interval holding 95 % of the low-minus-low differences (not the device band of section 2).

We run that protocol, a hundred ensembles per set and five repetitions, on six statistics:

- the **second-harmonic PLV** $|\langle e^{2i\delta}\rangle|$, the classical statistic of the moment that differs;
- its **square**, a check explained below;
- the QMI of the three weightings of section 2, and of a fourth, $(1, 0, 2)$, in which the first harmonic is switched off entirely.

In [ ]:
STATISTICS = {
    "second-harmonic PLV": lambda ta, tb: harmonic_plv(ta, tb, harmonic=2),
    "same, squared": lambda ta, tb: harmonic_plv(ta, tb, harmonic=2) ** 2,
    "QMI (1, 1, 1)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 1.0, 1.0)),
    "QMI (1, 0.5, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.5, 2.0)),
    "QMI (1, 0.2, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.2, 2.0)),
    "QMI (1, 0, 2)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.0, 2.0)),
}
N_RECORDS = 100  # ensembles per set, as in 04/16
N_SEEDS = 5      # independent repetitions of the whole comparison
SIZES = [50, 100, 200, 2_000, 20_000]  # samples per ensemble
FALSE_ALARM_RATE = 0.05  # fraction of null differences allowed outside the detection band

# matched_plv_ensembles consumes seed, seed + 1 and seed + 2. Every call gets its own multiple
# of 10, drawn without replacement, so no two calls share a random stream.
seed_table = 10 * np.random.default_rng(17).choice(
    10**8, size=(len(SIZES), N_SEEDS, 2, N_RECORDS), replace=False
)


def run_comparison(n, seeds_observed, seeds_null, statistics):
    """Protocol of 04/16: N_RECORDS (low, high) pairs and N_RECORDS (low, low) pairs.

    Returns, for each statistic, the area under the curve (probability that a high ensemble
    scores above a low one) and the detection rate (share of high - low differences outside
    the band that holds 95 % of the low - low differences).
    """
    values = {name: np.empty((4, N_RECORDS)) for name in statistics}  # low, high, null 1, null 2
    for i in range(N_RECORDS):
        low, high = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seeds_observed[i])
        )
        null_1, null_2 = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.0, seed=int(seeds_null[i])
        )
        for name, statistic in statistics.items():
            values[name][:, i] = [statistic(*pair) for pair in (low, high, null_1, null_2)]
    return {
        name: (
            auc(v[0], v[1]),
            detection_rate(v[1] - v[0], v[3] - v[2], false_alarm_rate=FALSE_ALARM_RATE),
        )
        for name, v in values.items()
    }


def print_tables(summary, sizes, title):
    """Print mean [min, max] over repetitions, one row per statistic and one column per size."""
    names = list(next(iter(summary.values())))
    for column, label in ((1, "detection rate"), (0, "area under the curve")):
        print(f"\n{title}: {label}, mean [min, max] over {N_SEEDS} repetitions")
        print(f"{'samples per ensemble':<22}" + "".join(f"{n:>19,}" for n in sizes))
        for name in names:
            cells = [summary[n][name][:, column] for n in sizes]
            print(f"{name:<22}" + "".join(
                f"{f'{c.mean():.2f} [{c.min():.2f}, {c.max():.2f}]':>19}" for c in cells
            ))


summary = {}
for k, n in enumerate(SIZES):
    start = time.perf_counter()
    runs = [
        run_comparison(n, seed_table[k, s, 0], seed_table[k, s, 1], STATISTICS)
        for s in range(N_SEEDS)
    ]
    summary[n] = {name: np.array([run[name] for run in runs]) for name in STATISTICS}
    print(f"{n:>7,} samples per ensemble: {N_SEEDS} repetitions done "
          f"in {time.perf_counter() - start:5.1f} s")
print_tables(summary, SIZES, "weightings")

**Read the output.** Each cell is a mean over five repetitions, lowest and highest in brackets. In the detection-rate table, the equal-weight QMI is at $0.15$ with 2,000 samples and still at $0.58$ with 20,000. With the first-harmonic weight at $0.5$ the rate is $0.80$ at 200 samples. At $0.2$ it is $0.88$ at 100 samples and $1.00$ at 200. The weighted measure does better with 100 samples than the equal-weight one with 20,000: on this pair of ensembles, one weight divided the samples needed by at least two hundred.

The second-harmonic PLV, at the top, is at $0.90$ with 100 samples and $0.99$ with 200: the weighted QMI has caught up with it. Has it gone past? The rows are too close to tell at a glance.

**How to read a near-tie.** Go to the 50-sample column, where nothing is saturated. By AUC the classical statistic is level or slightly ahead: $0.96$ (repetitions from $0.95$ to $0.97$) against $0.94$ ($0.92$ to $0.95$) for $(1, 0.2, 2)$. By detection rate the two are level, $0.57$ against $0.58$.

The squared row tells you which criterion to trust. Squaring cannot change the order of two values, so its AUC equals that of the PLV. Its detection rate is *higher*, $0.67$ against $0.57$. The detection rate works on differences, and a scale that compresses small values shrinks the low-minus-low differences more than a high-minus-low one. A statistic can gain detection rate from its scale alone; its AUC cannot move that way. In a near-tie, read the AUC first.

The weighted QMI has such a scale: near zero it grows like the *square* of the second moment, as section 4 derives. The next cell reads the bench of `04/17` to see what that scale does across many settings.

In [ ]:
# The bench of 04/17 was run once and its results are committed with the course, at the
# repository root, which we find by walking up from this notebook to the project file.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
with open(REPO_ROOT / "results" / "second_harmonic_bench" / "summary.csv", newline="") as handle:
    bench_rows = [row for row in csv.DictReader(handle) if row["experiment"] != "trap"]

# One setting = one (experiment, label, record length); one row per statistic in each.
settings = {}
for row in bench_rows:
    settings.setdefault((row["experiment"], row["label"], row["duration_s"]), {})[row["statistic"]] = row

print("bench of 04/17 (trap experiment left out): weighted QMI minus second-harmonic PLV")
for column, weights in (("qmi_weighted_02", "(1, 0.2, 2)"), ("qmi_weighted_05", "(1, 0.5, 2)")):
    for field, label in (("detection_rate_mean", "detection rate"), ("auc_mean", "AUC")):
        diff = np.array([float(st[column][field]) - float(st["plv2"][field])
                         for st in settings.values() if column in st and "plv2" in st])
        print(f"  {weights:<12} {label:<15} settings {diff.size:>3}: above {np.sum(diff > 0):>2}, "
              f"level {np.sum(diff == 0):>2}, below {np.sum(diff < 0):>2};  "
              f"largest above {diff.max():+.3f}, largest below {diff.min():+.3f}")

**Read the output.** In the bench files of `04/17`, the weights $(1, 0.2, 2)$ show a detection rate above that of the second-harmonic PLV in 14 of 24 settings, by up to $0.14$, and an AUC level or up to $0.025$ below in 22 of them (the other 2 are at most $0.002$ above). The weights $(1, 0.5, 2)$ fall below it in 14 settings on both criteria. Read with the AUC first: caught up, not ahead.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

styles = {
    "second-harmonic PLV": (COLORS["source"], "-"),
    "QMI (1, 1, 1)": (COLORS["muted"], "-"),
    "QMI (1, 0.5, 2)": (COLORS["quantum"], "-"),
    "QMI (1, 0.2, 2)": (COLORS["highlight"], "-"),
    "QMI (1, 0, 2)": (COLORS["text"], "--"),
}
POWER_CONVENTION = 0.8  # usual convention for statistical power, not derived from this problem
panels = [(0, "area under the curve", 0.5, "chance"),
          (1, "detection rate", POWER_CONVENTION, "0.8, the usual power convention")]
for ax, (column, ylabel, reference, reference_label) in zip(axes, panels):
    ax.axhline(reference, color=COLORS["muted"], lw=1.2, ls=":", zorder=1, label=reference_label)
    for name, (color, linestyle) in styles.items():
        table = np.array([summary[n][name][:, column] for n in SIZES])  # (sizes, repetitions)
        ax.fill_between(SIZES, table.min(axis=1), table.max(axis=1), color=color, alpha=0.18,
                        zorder=2)
        ax.plot(SIZES, table.mean(axis=1), color=color, ls=linestyle, lw=2.0, marker="o",
                ms=5, zorder=3, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("samples per ensemble")
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.03, 1.03)
axes[1].legend(loc="center right", fontsize=8)
fig.suptitle("Samples needed to tell the two ensembles apart, by weighting", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** The same numbers as curves, AUC on the left and detection rate on the right, against the number of samples on a logarithmic axis; each shaded band spans the five repetitions. The grey equal-weight curve climbs slowly and has not reached the dotted $0.8$ line at 20,000 samples. The blue curve (weight $0.5$) reaches it at 200 samples. The rose curve (weight $0.2$) and the dashed dark curve (weight $0$) lie on the violet curve of the second-harmonic PLV and partly hide it. Lowering the weight moves the quantum measure leftwards, towards the classical curve, and the curves stop there.

## 4. Why it works, and where it stops

On a longer and longer record a statistic settles to a fixed value: its **long-run** value. For these embeddings the long-run QMI depends on the data through two numbers only, the circular moments $m_1$ and $m_2$ of section 1; the modulus of $m_1$ is the PLV, and that of $m_2$ is the second-harmonic PLV. In our ensembles both are real and positive. `long_run_statistics` computes the long-run value from the moments themselves, with no sampling, so we can draw how each weighting responds to each moment.

Our comparison asks one thing of a measure: respond to $m_2$, which goes from $0$ to $0.3$. It asks it *not* to respond to $m_1$, which is $0.4$ in both ensembles and wanders around that value on any finite record. The next cell draws both responses.

In [ ]:
# long_run_statistics computes the measure from the moments themselves (the limit of an
# infinitely long record), so these curves carry no sampling noise. Moments are passed as
# (1, m1, m2), here real and positive.
WEIGHT_SETS = {
    "(1, 1, 1)": (1.0, 1.0, 1.0),
    "(1, 0.5, 2)": (1.0, 0.5, 2.0),
    "(1, 0.2, 2)": (1.0, 0.2, 2.0),
    "(1, 0, 2)": (1.0, 0.0, 2.0),
}


def long_run_qmi(m1, m2):
    """Long-run QMI (nats) of every weighting at first moment m1 and second moment m2."""
    stats = long_run_statistics(np.array([1.0, m1, m2], dtype=complex), weight_sets=WEIGHT_SETS)
    return np.array([stats[name] for name in WEIGHT_SETS])


m2_grid = np.linspace(0.0, 0.4, 41)
m1_grid = np.linspace(0.3, 0.5, 41)
along_m2 = np.array([long_run_qmi(0.4, m2) for m2 in m2_grid])   # (41, weightings)
along_m1 = np.array([long_run_qmi(m1, 0.15) for m1 in m1_grid])  # (41, weightings)

# Check against the ensembles of section 2: the long-run gap between m2 = 0 and m2 = 0.3.
print("long-run gap between m2 = 0.0 and m2 = 0.3 at m1 = 0.4, beside the gap of section 2")
for j, name in enumerate(WEIGHT_SETS):
    sampled = f"{gaps[j]:.5f}" if j < len(gaps) else "    -"
    print(f"  {name:<12} long run {along_m2[30, j] - along_m2[0, j]:.5f}   sampled {sampled}")

# The scale of the weighted measures near zero: double m2 and see by how much the long-run
# QMI, counted from its value at m2 = 0, is multiplied. A square law gives 4.
print("\nlong-run QMI counted from m2 = 0, at m1 = 0.4: effect of doubling m2")
print(f"{'weights':<14}{'m2 = 0.05':>11}{'m2 = 0.10':>11}{'m2 = 0.20':>11}{'ratios':>14}")
for j, name in enumerate(WEIGHT_SETS):
    rise = along_m2[[5, 10, 20], j] - along_m2[0, j]
    print(f"{name:<14}{rise[0]:>11.5f}{rise[1]:>11.5f}{rise[2]:>11.5f}"
          f"{rise[1] / rise[0]:>8.1f}{rise[2] / rise[1]:>6.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
curve_colors = [COLORS["muted"], COLORS["quantum"], COLORS["highlight"], COLORS["text"]]
curve_styles = ["-", "-", "-", "--"]
for j, name in enumerate(WEIGHT_SETS):
    axes[0].plot(m2_grid, along_m2[:, j] - along_m2[0, j], color=curve_colors[j],
                 ls=curve_styles[j], lw=2.0, label=name)
    axes[1].plot(m1_grid, along_m1[:, j] - along_m1[20, j], color=curve_colors[j],
                 ls=curve_styles[j], lw=2.0, label=name)
axes[0].axhline(0.0, color=COLORS["muted"], lw=1.0, zorder=1)
axes[1].axhline(0.0, color=COLORS["muted"], lw=1.0, zorder=1)
axes[0].set_xlabel("second moment $m_2$  (first moment fixed at 0.4)")
axes[0].set_ylabel("QMI minus its value at $m_2 = 0$  (nats)")
axes[0].set_title("What the comparison asks for", pad=10)
axes[1].set_xlabel("first moment $m_1$  (second moment fixed at 0.15)")
axes[1].set_ylabel("QMI minus its value at $m_1 = 0.4$  (nats)")
axes[1].set_title("What the comparison does not ask for", pad=10)
axes[0].legend(title="weights", loc="upper left", fontsize=9)
fig.suptitle("Long-run response of each weighting to the two moments", y=1.0)
fig.tight_layout()
plt.show()

**Read the output and the figure.** The first block checks the tool: the long-run gaps agree with the sampled ones of section 2 within $0.00015$ nat for the weighted embeddings, and by about $0.0005$ for the equal-weight one, which the next cell shows to be the most disturbed by a finite record. The second block checks the scale: for $(1, 0.2, 2)$ and $(1, 0, 2)$, doubling $m_2$ multiplies the rise by $4.0$, a square law.

*Left panel.* As $m_2$ grows, the three weighted curves rise from the start. The grey equal-weight curve first goes *down*, stays below its start until $m_2 \approx 0.17$, and only then rises: it spends part of the difference it is given on getting back to where it began.

*Right panel.* As $m_1$ moves, the equal-weight curve is steep, the blue one flatter, the rose one almost flat, and the dashed one exactly flat.

In [ ]:
# Slopes: how many nats the measure moves per unit of each moment, around m1 = 0.4, m2 = 0.15,
# from a step of 0.01 on each side.
slope_m2 = (long_run_qmi(0.4, 0.16) - long_run_qmi(0.4, 0.14)) / 0.02
slope_m1 = (long_run_qmi(0.41, 0.15) - long_run_qmi(0.39, 0.15)) / 0.02
print("slope of the measure at m1 = 0.4, m2 = 0.15  (nats per unit of moment)")
print(f"{'weights':<14}{'along m1':>10}{'along m2':>10}{'ratio m1 / m2':>16}")
for j, name in enumerate(WEIGHT_SETS):
    print(f"{name:<14}{slope_m1[j]:>10.4f}{slope_m2[j]:>10.4f}{slope_m1[j] / slope_m2[j]:>16.2f}")

# By hand: the sample PLV of a finite record wanders around 0.4. Multiply its scatter by the
# slope along m1 to get the scatter this wandering alone puts into each measure. Then measure
# the scatter of each measure over the same records, low and high, and set the larger of the
# two beside the long-run gap.
N_CHECK = 2_000
records = [
    matched_plv_ensembles(N_CHECK, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seed))
    for seed in seed_table[SIZES.index(N_CHECK), 0, 0]
]  # 100 (low, high) pairs of ensembles of 2,000 samples
plv_scatter = np.std([plv(*low) for low, _ in records], ddof=1)
print(f"\nscatter of the sample PLV over {len(records)} low ensembles of {N_CHECK:,} samples: "
      f"{plv_scatter:.4f}")
print(f"{'':<14}{'scatter from':>14}{'measured scatter':>20}{'long-run':>12}{'gap / larger':>14}")
print(f"{'weights':<14}{'the PLV':>14}{'low':>10}{'high':>10}{'gap':>12}{'scatter':>14}")
for j, (name, weights) in enumerate(WEIGHT_SETS.items()):
    from_plv = slope_m1[j] * plv_scatter
    low_scatter = np.std([weighted_qmi(*low, weights) for low, _ in records], ddof=1)
    high_scatter = np.std([weighted_qmi(*high, weights) for _, high in records], ddof=1)
    gap = along_m2[30, j] - along_m2[0, j]
    print(f"{name:<14}{from_plv:>14.5f}{low_scatter:>10.5f}{high_scatter:>10.5f}{gap:>12.5f}"
          f"{gap / max(low_scatter, high_scatter):>14.1f}")

**Read the output.** The **slope** is nats per unit of moment; the **scatter** is a standard deviation over the hundred records. The equal-weight measure moves by $0.37$ nat per unit of $m_1$, about twenty-three times its slope along $m_2$. With the weight at $0.2$ the slope along $m_1$ is $0.007$, a sixth of the slope along $m_2$. At $0$ it is zero, as section 1 predicts.

The second table sets a hand computation beside a measurement: the sample PLV of a 2,000-sample record scatters by $0.012$. Multiplied by the slope, that wandering alone puts $0.0046$ nat of scatter into the equal-weight QMI. The measured scatter is $0.0047$ nat for the low ensembles and $0.0056$ for the high ones: most of it comes from the matched moment, and it is as large as the whole gap ($0.0049$).

For $(1, 0.2, 2)$ the first moment passes on $0.00009$ nat only. What remains, $0.00015$ nat for the low ensembles and $0.0015$ for the high ones, is the scatter of the second moment itself, which no weighting removes. The gap is now about ten times the larger scatter, where it was a little under once.

**Where it stops, by hand.** Follow the weight to zero. With $w_1 = 0$ each channel is a qubit on the levels 0 and 2, with $p_0 + p_2 = 1$. The joint density is diagonal except for one coherence, $p_0p_2\,m_2$, between $|0,2\rangle$ and $|2,0\rangle$, whose diagonal entries are both $p_0p_2$. That $2\times2$ block has eigenvalues $p_0p_2\,(1 \pm |m_2|)$, and the marginals do not depend on $m_2$, so the mutual information is the drop in joint entropy the coherence causes:
$$
I = p_0p_2\,\Bigl[(1 + |m_2|)\ln(1 + |m_2|) + (1 - |m_2|)\ln(1 - |m_2|)\Bigr] \approx p_0p_2\,|m_2|^2 .
$$
Three facts about long records follow. The long-run value depends on $|m_2|$ alone, the second-harmonic PLV, and rises with it: it is that statistic on another scale, so no difference of ranking can come from the long-run function itself. Near zero it grows like $|m_2|^2$, the square law of section 3. And the weights enter as one factor, $p_0p_2$; for $(1, 0, 2)$ it is $0.16$.

**A worked counter-example: a larger gap that detects no better.** The factor $p_0p_2$ is largest at $p_0 = p_2 = \tfrac12$, the weights $(1, 0, 1)$: $0.25$ instead of $0.16$. If the size of the gap were what counts, $(1, 0, 1)$ would be the better design. The closed form says the two are the same function of $|m_2|$ up to a constant, so whatever separates them on a record of finite length comes from the entries that the closed form drops, not from the size of the gap. The next cell checks the closed form against `long_run_statistics` and runs the protocol of section 3 on both, on the same records.

In [ ]:
def closed_form_qmi(p0, m2):
    """Long-run QMI (nats) of the weights (w0, 0, w2) with p0 = w0^2 / (w0^2 + w2^2)."""
    m = abs(m2)
    return p0 * (1 - p0) * ((1 + m) * np.log1p(m) + (1 - m) * np.log1p(-m))


ZERO_SETS = {"(1, 0, 2)": (1.0, 0.0, 2.0), "(1, 0, 1)": (1.0, 0.0, 1.0)}
P0 = {"(1, 0, 2)": 0.2, "(1, 0, 1)": 0.5}  # w0^2 / (w0^2 + w2^2)
bracket = (1.3 * np.log(1.3) + 0.7 * np.log(0.7))
print(f"bracket at |m2| = 0.3: {bracket:.4f}")
curves = {}
for name, w in ZERO_SETS.items():
    p0 = P0[name]
    tool = [long_run_statistics(np.array([1.0, 0.4, m2], dtype=complex),
                                weight_sets={name: w})[name] for m2 in m2_grid]
    curves[name] = (closed_form_qmi(p0, m2_grid), np.array(tool))
    print(f"{name}: p0 p2 = {p0 * (1 - p0):.2f}   long-run gap by hand {closed_form_qmi(p0, 0.3):.5f}"
          f"   from long_run_statistics {tool[30] - tool[0]:.5f}")
print(f"ratio of the two gaps: {closed_form_qmi(0.5, 0.3) / closed_form_qmi(0.2, 0.3):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for name, color in (("(1, 0, 2)", COLORS["text"]), ("(1, 0, 1)", COLORS["flow"])):
    by_hand, tool = curves[name]
    axes[0].plot(m2_grid, by_hand, color=color, lw=2.0, label=f"{name}, closed form")
    axes[0].plot(m2_grid[::4], tool[::4], "o", color=color, ms=6, mfc="white",
                 label=f"{name}, long_run_statistics")
    axes[1].plot(m2_grid, by_hand / by_hand[30], color=color, lw=2.0, label=name)
axes[1].plot(m2_grid, (m2_grid / 0.3) ** 2, color=COLORS["muted"], ls=":", lw=1.4,
             label="$(|m_2| / 0.3)^2$")
axes[0].set_xlabel("second moment $|m_2|$")
axes[0].set_ylabel("long-run QMI  (nats)")
axes[0].set_title("Two weightings, two scales", pad=10)
axes[0].legend(loc="upper left", fontsize=8)
axes[1].set_xlabel("second moment $|m_2|$")
axes[1].set_ylabel("long-run QMI / its value at $|m_2| = 0.3$")
axes[1].set_title("One shape", pad=10)
axes[1].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

# The closed form keeps only the entries with i - k = l - j (the others average to zero when
# theta_A is exactly uniform). KEEP is 1 on those entries and 0 elsewhere; a "masked" row sets
# the dropped entries of the sample density to zero before taking the QMI.
LEVELS = [(i, j) for i in range(3) for j in range(3)]  # row 3 i + j is |i>_A |j>_B
KEEP = np.array([[float(i - k == l - j) for (k, l) in LEVELS] for (i, j) in LEVELS])


def masked_qmi(ta, tb, weights):
    """QMI (nats) of the weighted embedding with the entries the closed form drops set to zero."""
    return coupling_qmi(joint_density(ta, tb, weights) * KEEP, dims=(3, 3))


# The protocol of section 3 on the same records (same rows of seed_table), short records only.
COUNTER_SIZES = [50, 100, 200]
COUNTER_STATISTICS = {
    "second-harmonic PLV": STATISTICS["second-harmonic PLV"],
    "same, squared": STATISTICS["same, squared"],
    "QMI (1, 0, 2)": STATISTICS["QMI (1, 0, 2)"],
    "QMI (1, 0, 1)": lambda ta, tb: weighted_qmi(ta, tb, (1.0, 0.0, 1.0)),
    "masked (1, 0, 2)": lambda ta, tb: masked_qmi(ta, tb, (1.0, 0.0, 2.0)),
    "masked (1, 0, 1)": lambda ta, tb: masked_qmi(ta, tb, (1.0, 0.0, 1.0)),
}
counter_summary = {}
for n in COUNTER_SIZES:
    k = SIZES.index(n)
    runs = [run_comparison(n, seed_table[k, s, 0], seed_table[k, s, 1], COUNTER_STATISTICS)
            for s in range(N_SEEDS)]
    counter_summary[n] = {name: np.array([run[name] for run in runs]) for name in COUNTER_STATISTICS}
print_tables(counter_summary, COUNTER_SIZES, "counter-example")

# Paired differences: both statistics are scored on the same records in each repetition.
print("\nAUC differences on the same records, one value per repetition")
for n in (50, 100):
    for a, b in (("QMI (1, 0, 1)", "QMI (1, 0, 2)"), ("QMI (1, 0, 2)", "second-harmonic PLV")):
        diff = counter_summary[n][a][:, 0] - counter_summary[n][b][:, 0]
        print(f"  {n:>4} samples  {a} minus {b}: " + "  ".join(f"{d:+.4f}" for d in diff))

# How large are the dropped entries on a short record? Low ensembles of 50 samples, first
# repetition, weights (1, 0, 2): the largest dropped entry against the entry that carries m2.
dropped, kept_m2 = [], []
for seed in seed_table[SIZES.index(50), 0, 0]:
    (ta, tb), _ = matched_plv_ensembles(50, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seed))
    rho = joint_density(ta, tb, (1.0, 0.0, 2.0))
    dropped.append(np.max(np.abs(rho * (1 - KEEP))))
    kept_m2.append(abs(rho[M2_ENTRY]))
print(f"\nlow ensembles of 50 samples, weights (1, 0, 2): median largest dropped entry "
      f"{np.median(dropped):.4f}, median |<2,0|rho|0,2>| {np.median(kept_m2):.4f}, "
      f"ratio {np.median(dropped) / np.median(kept_m2):.1f}")

**Read the output and the figure.** The closed form gives long-run gaps of $0.01462$ and $0.02285$ nat, the values of `long_run_statistics` to the fifth decimal, a ratio of $1.5625 = 0.25/0.16$.

*Left panel.* Lines (closed form) and circles (library) coincide; the green curve of $(1, 0, 1)$ sits above the dark one. *Right panel.* Divided by their values at $|m_2| = 0.3$, the two collapse onto one curve, with the dotted square law under them. On long records they are one statistic on two scales.

*The tables.* On the records of section 3, the larger gap does not detect better. At 50 samples $(1, 0, 1)$ has a detection rate of $0.51$ against $0.57$ for $(1, 0, 2)$. On the same records, $(1, 0, 1)$ ranks below $(1, 0, 2)$ in each of the five repetitions at 50 samples, by $0.008$ to $0.014$ in AUC; at 100 samples the difference is within $0.003$. The gap grew by half, and the ranking did not improve.

Why do both rank below the second-harmonic PLV at 50 samples ($0.94$ and $0.93$ against $0.96$), when in the long run they are functions of it? On 50 samples the entries with $i - k \neq l - j$ are not small: in a low ensemble the largest of them is about $2.7$ times the entry that carries $m_2$ (medians $0.0495$ and $0.0182$). With those entries set to zero (the masked rows), both weightings give back the AUC of the second-harmonic PLV and the detection rate of its square, in every column: those entries are the whole of the difference.

## 5. What the dial buys

The weights brought the quantum measure level with the second-harmonic PLV, which in the long run it becomes at $w_1 = 0$, on another scale. Weighting changed how many samples the measure needs. It did not make it read something the second-harmonic PLV misses, or rank records better than that statistic.

That is the conclusion of `04/17`, seen from the side of design: find the moment that carries your difference, and give it the weight. Keep the two yardsticks apart: the gap in nats is what a device must resolve, the AUC is what the samples allow, and section 4 shows the gap can grow while the AUC stays put.

**Choose the weights without peeking.** The weights here were picked knowing which moment differs, and scored on the same ensembles. That is circular and flatters the choice. On your data, fix the weights from a hypothesis stated in advance (a coupling expected in the second harmonic means, by section 1, a small $w_1$), or choose them on part of the records and score them on the rest, as the fourth exercise asks.

**Limits.** One pair of ensembles; samples drawn independently, where real signals are correlated in time (the bench of `04/17` covers that case); five weightings, not an optimisation; the $0.8$ line and the two-or-three-bands rule are conventions. `04/20` asks whether *any* embedding of classical data could cross this boundary.

This notebook earns one row of the classical-to-quantum dictionary. It sets the second-harmonic PLV beside the weighted embedding that section 4 showed to become that statistic, on another scale, when the first harmonic is switched off.

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| second-harmonic PLV $\lvert\langle e^{2i\delta}\rangle\rvert$ | weighted qutrit embedding: with $w_1 = 0$ its long-run QMI is a function of that statistic alone; a small $w_1$ brings the QMI level with the statistic on short records, and no further |

## Your turn

**1. Warm-up.** Add the weighting $(1, 0.1, 2)$ to `STATISTICS` and `WEIGHT_SETS`. Before running, predict its slope along $m_1$ and the rows its detection rate will fall between at 50 and 100 samples. (Use the factor that section 1 found in every entry carrying $m_1$, and the slope printed for $w_1 = 0.2$.)

**2. Go further: more harmonics.** `weighted_qmi(ta, tb, None, (1, 2, 3))` is the equal-weight embedding with four levels. Our ensembles have no third or fourth moment. Predict whether the added levels help anyway, then add the harmonics $(1, 2, 3)$ and $(1, 2, 3, 4)$ to `STATISTICS` and read the 2,000-sample column. (Your run prints the new rows beside the equal-weight row and the second-harmonic PLV: check your prediction in the 2,000-sample column. Then count, as in section 1, the pairs of levels two steps apart, and say whether your numbers prove that this is the cause.)

**3. The largest gap, by hand.** For the weights $(1, 0, w_2)$, show that $p_0p_2$ is largest at $w_2 = 1$. Predict with the closed form the long-run gap of $(1, 0, 3)$ between $m_2 = 0$ and $0.3$, check it with `long_run_statistics`, then choose a third value of $w_2$, run the protocol at 50 samples beside $(1, 0, 2)$ and $(1, 0, 1)$, and say whether the AUC follows $p_0p_2$.

**4. Challenge: choose without peeking.** At 50 samples, use the first 50 pairs of one repetition to pick, among `WEIGHT_SETS`, the weighting with the highest AUC; score that choice alone on the other 50 pairs, beside the second-harmonic PLV on the same pairs. Repeat over the five repetitions and write the sentence your report would carry.

## What you built

You turned the embedding into an instrument you can tune, and measured every turn of the dial. That careful work is yours.

- You can write by hand which entries of $\rho_{AB}$ carry each moment, and see on the matrix that $w_1$ quietens the first.
- You can measure the gap between the ensembles of `04/16` grow about threefold, to a little under one band of a real device.
- You can show with the protocol of `04/16` that weighting divides the samples needed by at least two hundred, down to the level of the second-harmonic PLV, and no further.
- You can read a near-tie: the AUC ignores the scale of a statistic, the detection rate does not.
- You can derive the closed form at $w_1 = 0$, and show with $(1, 0, 1)$ that a larger gap is not a better detector.

## What's next

Every embedding here writes phases only. In `19_amplitude_embedding_and_phase_amplitude_coupling` you score a lever of another kind: an embedding that also takes the amplitude of the signal and reaches a coupling no phase statistic can see, set beside the classical statistic built for it.

## References

- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments; the weighting tunes which moment the coupling measure emphasises.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, the first-moment statistic whose channel we turn down.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information, the coupling measure read off $\rho_{AB}$.
- J. A. Hanley & B. J. McNeil, "The meaning and use of the area under a receiver operating characteristic (ROC) curve", *Radiology* **143**, 29–36 (1982). DOI:10.1148/radiology.143.1.7063747 — the area under the curve, read as a probability of correct ranking.

**Previous:** `notebooks/04_QuantumOptimalTransport/17_capstone_synthesis.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/19_amplitude_embedding_and_phase_amplitude_coupling.ipynb`